<a href="https://colab.research.google.com/github/ZehanQin/ECON5200-Applied-Data-Analytics-in-Econ/blob/main/Lab%2012/Lab_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

url = 'https://raw.githubusercontent.com/ZehanQin/ECON5200-Applied-Data-Analytics-in-Econ/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)
df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [3]:
formula = 'Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [6]:
model = smf.ols(formula = formula, data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Tue, 10 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        18:56:07   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [7]:
y_pred = results.predict(df)

In [10]:
model_rmse = rmse(df['Home_Value'], y_pred)
print(f"\nThe Predictive RMSE is: $ {model_rmse:,.2f}")


The Predictive RMSE is: $ 42,316.69


In [11]:
"""
Residual Forensics Dashboard for Hedonic Pricing OLS Model
===========================================================
Builds an interactive Plotly scatter plot of Fitted Values vs. Residuals,
highlights outliers beyond ±2 SD, and overlays a zero-reference line.

Prerequisites:
    pip install statsmodels plotly pandas numpy scikit-learn
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go
from sklearn.datasets import fetch_california_housing

# ============================================================
# 1. LOAD DATA & FIT HEDONIC PRICING OLS MODEL
# ============================================================
# Using California Housing as a proxy hedonic dataset.
# Each feature (MedInc, HouseAge, etc.) represents a housing
# "characteristic" whose implicit price the model estimates.
housing = fetch_california_housing(as_frame=True)
X = housing.frame.drop(columns="MedHouseVal")
y = housing.frame["MedHouseVal"]

# Add a constant (intercept) column — required by statsmodels OLS
X_const = sm.add_constant(X)

# Fit the OLS model
ols_model = sm.OLS(y, X_const).fit()

# Quick sanity check — print the regression summary to console
print(ols_model.summary())

# ============================================================
# 2. EXTRACT RESIDUALS & FITTED VALUES FROM THE RESULTS OBJECT
# ============================================================
# .fittedvalues → pd.Series of ŷ (predicted y for each observation)
# These are the model's in-sample predictions: X @ β̂
fitted_vals = ols_model.fittedvalues

# .resid → pd.Series of ê = y − ŷ (raw OLS residuals)
# Positive residual = model under-predicted; Negative = over-predicted
residuals = ols_model.resid

# ============================================================
# 3. CALCULATE RMSE (Root Mean Squared Error)
# ============================================================
# RMSE gives the average magnitude of prediction error in the
# same units as the dependent variable (here: $100,000s).
rmse = np.sqrt(np.mean(residuals ** 2))
print(f"\nRMSE: {rmse:.4f}  (units: $100k)")

# ============================================================
# 4. IDENTIFY OUTLIERS BEYOND ±2 STANDARD DEVIATIONS
# ============================================================
# Compute the standard deviation of the residual distribution.
resid_std = residuals.std()

# Boolean mask: True where |ê| > 2σ — these are potential
# influential points, data-entry errors, or structural mismatches.
outlier_mask = residuals.abs() > 2 * resid_std

# Build a label column: every point is tagged for the legend.
labels = np.where(outlier_mask, "Outlier (|e| > 2σ)", "Normal")

print(f"Residual σ: {resid_std:.4f}")
print(f"Outliers detected: {outlier_mask.sum()} / {len(residuals)} "
      f"({outlier_mask.mean() * 100:.1f}%)")

# ============================================================
# 5. ASSEMBLE A TIDY DATAFRAME FOR PLOTLY
# ============================================================
plot_df = pd.DataFrame({
    "Fitted Values (ŷ)": fitted_vals,
    "Residuals (e)": residuals,
    "Category": labels,
    # Absolute residual shown on hover for quick magnitude check
    "|Residual|": residuals.abs().round(4),
})

# ============================================================
# 6. BUILD THE INTERACTIVE SCATTER PLOT
# ============================================================
# color_discrete_map pins the two category labels to fixed colours
# so "Outlier" is always crimson regardless of sort order.
fig = px.scatter(
    plot_df,
    x="Fitted Values (ŷ)",
    y="Residuals (e)",
    color="Category",
    color_discrete_map={
        "Normal": "#636EFA",           # Plotly's default blue
        "Outlier (|e| > 2σ)": "#DC143C"  # Stark crimson for outliers
    },
    hover_data={"|Residual|": True, "Category": False},
    opacity=0.55,
    title="Residual Forensics Dashboard — Hedonic Pricing OLS",
    labels={
        "Fitted Values (ŷ)": "Fitted Predicted Values (ŷ)",
        "Residuals (e)": "Residual Error (e = y − ŷ)"
    },
)

# --- Horizontal zero-line: the ideal residual target --------
# If the model were perfect, every point would sit on this line.
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="white",
    line_width=1.5,
    annotation_text="Zero Residual",
    annotation_font_color="white",
    annotation_position="bottom right",
)

# --- ±2σ boundary bands (optional visual guides) ------------
# These dashed lines bracket the "expected" residual range.
for bound, tag in [(2 * resid_std, "+2σ"), (-2 * resid_std, "−2σ")]:
    fig.add_hline(
        y=bound,
        line_dash="dot",
        line_color="#DC143C",
        line_width=1,
        annotation_text=tag,
        annotation_font_color="#DC143C",
        annotation_position="bottom right",
    )

# --- Dark theme for readability & presentation quality ------
fig.update_layout(
    template="plotly_dark",
    height=620,
    width=1050,
    legend_title_text="Residual Category",
    xaxis_title="Fitted Predicted Values (ŷ)",
    yaxis_title="Residual Error (e = y − ŷ)",
    font=dict(family="Inconsolata, monospace", size=13),
    margin=dict(t=70, b=50),
)

# ============================================================
# 7. RENDER
# ============================================================
fig.show()

                            OLS Regression Results                            
Dep. Variable:            MedHouseVal   R-squared:                       0.606
Model:                            OLS   Adj. R-squared:                  0.606
Method:                 Least Squares   F-statistic:                     3970.
Date:                Tue, 10 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:32:56   Log-Likelihood:                -22624.
No. Observations:               20640   AIC:                         4.527e+04
Df Residuals:                   20631   BIC:                         4.534e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -36.9419      0.659    -56.067      0.0